In [1]:
from scripts.get_log_embed import download_log, clear_log
from timeline.ingest.embed import import_embed_csv
from timeline.storage.repository import SQLiteRepository
from timeline.domain.interval_builder import IntervalBuilder
from pathlib import Path

In [3]:
import os
print(os.getcwd())

/home/dedzuka/Projects/behaviour_hub/notebooks


In [2]:
ip = input()
download_log(ip=ip, out_path=Path("../data/embed/incoming.csv"))
clear_log(ip=ip)

Fetching http://192.168.3.55/log.csv ...
Saved 1518 bytes to ../data/embed/incoming.csv
Sending DELETE to http://192.168.3.55/log.csv ...
log.csv cleared


In [3]:
with SQLiteRepository(Path("../timeline.db")) as repo:
    repo.ensure_schema()
    events = import_embed_csv(Path("../data/embed/incoming.csv"))
    for e in events:
        repo.insert_event(e)

In [8]:
with SQLiteRepository(Path("../timeline.db")) as repo:
        events = repo.load_events()
        chunks, warnings = IntervalBuilder.build(events)
        repo.replace_chunks(chunks)
        
        for warning in warnings:
            print(f"WARNING: {warning}")

print(f"Generated {len(chunks)} chunks.")

Generated 43 chunks.
